# Phase 8 — Imbalance Handling and Hyperparameter Tuning

## Objective

This phase evaluates the tuned Random Forest with a strict validation protocol. The primary metric is PR-AUC / average precision because fraud is highly imbalanced.

## Leakage-control rule

The five-fold validation is performed on `X_train` and `y_train` only. The test set is evaluated only after the training-only validation has finished. The readiness decision is not based on test performance.

## Execution

Run the cells sequentially from top to bottom using the local Python kernel in VS Code.

## 1. Configure the local project path

In [1]:
from pathlib import Path

PROJECT = Path.cwd().resolve()
PROJECT = PROJECT.parent if PROJECT.name == "notebooks" else PROJECT
PROCESSED = PROJECT / 'data' / 'processed'
MODELS = PROJECT / 'models'
RESULTS = PROJECT / 'results'

if not (PROCESSED / 'X_train_processed.npz').exists():
    raise FileNotFoundError(f'لم يتم العثور على ملفات البيانات داخل: {PROCESSED}')

MODELS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)
print('Project:', PROJECT)
print('Processed:', PROCESSED)


Project: /home/ubuntu/fraud_local
Processed: /home/ubuntu/fraud_local/data/processed


## 2. Import libraries and load the processed data

In [2]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate


PROJECT = Path.cwd().resolve()
PROJECT = PROJECT.parent if PROJECT.name == "notebooks" else PROJECT
PROCESSED = PROJECT / "data" / "processed"
MODELS = PROJECT / "models"
RESULTS = PROJECT / "results"
MODELS.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

print("PHASE 8 FAST RUN", flush=True)
print("Project:", PROJECT, flush=True)


PHASE 8 FAST RUN


Project: /home/ubuntu/fraud_local


## 3. Execute true five-fold training-only validation

In [3]:
X_train = sparse.load_npz(PROCESSED / "X_train_processed.npz")
X_test = sparse.load_npz(PROCESSED / "X_test_processed.npz")
y_train = pd.read_csv(PROCESSED / "y_train.csv")["isFraud"].to_numpy()
y_test = pd.read_csv(PROCESSED / "y_test.csv")["isFraud"].to_numpy()
phase6 = pd.read_csv(RESULTS / "model_comparison.csv")
print("X_train:", X_train.shape, "X_test:", X_test.shape, flush=True)
print("Test is loaded but will be used only for final evaluation.", flush=True)


X_train: (472432, 495) X_test: (118108, 495)


Test is loaded but will be used only for final evaluation.


## 4. Define evaluation metrics

In [4]:
scoring = {
    "pr_auc": "average_precision",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "roc_auc": "roc_auc",
}
cv_search = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
cv_final = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

counts = pd.Series(y_train).value_counts().sort_index()
shares = pd.Series(y_train).value_counts(normalize=True).sort_index()
ratio = float((y_train == 0).sum() / (y_train == 1).sum())
print("Fraud count:", int(counts.get(1, 0)), "Fraud %:", float(shares.get(1, 0) * 100), flush=True)
print("Imbalance ratio:", ratio, flush=True)

rf_row = phase6.loc[phase6["Model"] == "Random Forest"].iloc[0]
print("Historical Random Forest PR-AUC:", float(rf_row["PR-AUC"]), flush=True)

# Compare the two simple class-weight strategies using training folds only.
weight_rows = []
for label, weight in [("None", None), ("Balanced", "balanced")]:
    print("Starting class-weight CV:", label, flush=True)
    model = RandomForestClassifier(
        n_estimators=20,
        random_state=42,
        n_jobs=-1,
        class_weight=weight,
    )
    scores = cross_validate(model, X_train, y_train, cv=cv_search, scoring=scoring, n_jobs=1)
    row = {"Strategy": label}
    row.update({key: float(np.mean(scores["test_" + key])) for key in scoring})
    weight_rows.append(row)

weight_table = pd.DataFrame(weight_rows)
print("Class-weight results:", flush=True)
print(weight_table.to_string(index=False), flush=True)
selected_strategy = weight_table.sort_values(["pr_auc", "f1"], ascending=False).iloc[0]["Strategy"]
selected_weight = None if selected_strategy == "None" else "balanced"
print("Selected strategy:", selected_strategy, flush=True)

base = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    class_weight=selected_weight,
)
param_distributions = {
    "n_estimators": [20, 30, 50],
    "max_depth": [8, 12, 16, None],
    "min_samples_leaf": [1, 2, 5],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", 0.5],
}

print("Starting RandomizedSearchCV: 3 folds x 3 settings", flush=True)
start = time.time()
search = RandomizedSearchCV(
    base,
    param_distributions,
    n_iter=3,
    scoring=scoring,
    refit="pr_auc",
    cv=cv_search,
    random_state=42,
    n_jobs=1,
    return_train_score=False,
    verbose=1,
)
search.fit(X_train, y_train)
print("Search seconds:", round(time.time() - start, 2), flush=True)
print("Best parameters:", search.best_params_, flush=True)
print("Best 3-fold PR-AUC:", float(search.cv_results_["mean_test_pr_auc"][search.best_index_]), flush=True)

best_i = search.best_index_
cv_summary = {
    key: float(search.cv_results_["mean_test_" + key][best_i]) for key in scoring
}
cv_std = {
    key: float(search.cv_results_["std_test_" + key][best_i]) for key in scoring
}

# Final training uses all training rows; test remains untouched until the next block.
tuned_model = search.best_estimator_
tuned_model.fit(X_train, y_train)
print("Tuned model trained on complete X_train.", flush=True)

# True final validation on training data only. The test set is not used here.
print("Starting true 5-fold training-only validation.", flush=True)
cv_final_scores = cross_validate(tuned_model, X_train, y_train, cv=cv_final, scoring=scoring, n_jobs=1, return_train_score=False)
cv_final_summary = {key: float(np.mean(cv_final_scores["test_" + key])) for key in scoring}
cv_final_std = {key: float(np.std(cv_final_scores["test_" + key], ddof=1)) for key in scoring}
print("5-fold validation summary:", cv_final_summary, flush=True)
print("5-fold validation standard deviations:", cv_final_std, flush=True)


Fraud count: 16530 Fraud %: 3.498916246147594


Imbalance ratio: 27.580278281911674


Historical Random Forest PR-AUC: 0.7311031784295605


Starting class-weight CV: None


Starting class-weight CV: Balanced


Class-weight results:


... (7 outputs left out)

Tuned model trained on complete X_train.


Starting true 5-fold training-only validation.


5-fold validation summary: {'pr_auc': 0.7048871268268885, 'f1': 0.6666827372347159, 'recall': 0.608227465214761, 'precision': 0.737666506916508, 'roc_auc': 0.9318181610289423}


5-fold validation standard deviations: {'pr_auc': 0.00898912674359926, 'f1': 0.007883573941511473, 'recall': 0.01064154892923394, 'precision': 0.007505758926284348, 'roc_auc': 0.0033927544224974615}


## 5. Evaluate the tuned model on the untouched test set

In [5]:
def metrics(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y, pred).ravel()
    return {
        "Accuracy": float(accuracy_score(y, pred)),
        "Balanced Accuracy": float(balanced_accuracy_score(y, pred)),
        "Precision": float(precision_score(y, pred, zero_division=0)),
        "Recall": float(recall_score(y, pred, zero_division=0)),
        "F1": float(f1_score(y, pred, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y, proba)),
        "PR-AUC": float(average_precision_score(y, proba)),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }

print("Evaluating final tuned model on X_test only now.", flush=True)
tuned_test = metrics(tuned_model, X_test, y_test)
untuned_model = joblib.load(MODELS / "random_forest.pkl")
untuned_test = metrics(untuned_model, X_test, y_test)
print("Tuned test metrics:", tuned_test, flush=True)
print("Untuned test metrics:", untuned_test, flush=True)

train_tuned = metrics(tuned_model, X_train, y_train)


Evaluating final tuned model on X_test only now.


Tuned test metrics: {'Accuracy': 0.9801537575778101, 'Balanced Accuracy': 0.8137818712286744, 'Precision': 0.7586007516623302, 'Recall': 0.6348899104766513, 'F1': 0.6912539515279241, 'ROC-AUC': 0.9372922354099518, 'PR-AUC': 0.7305175952567509, 'TN': 113140, 'FP': 835, 'FN': 1509, 'TP': 2624}


Untuned test metrics: {'Accuracy': 0.9809581061401429, 'Balanced Accuracy': 0.7672126290849011, 'Precision': 0.8682564503518374, 'Recall': 0.5373820469392693, 'F1': 0.6638768494993275, 'ROC-AUC': 0.94051273867316, 'PR-AUC': 0.7224630422250053, 'TN': 113638, 'FP': 337, 'FN': 1912, 'TP': 2221}


## 6. Save corrected metadata and report

In [6]:
# Readiness is based only on five-fold training validation, never on X_test.
improved = (
    cv_final_summary["pr_auc"] > 0.0
    and cv_final_summary["f1"] > 0.0
    and cv_final_std["pr_auc"] < cv_final_summary["pr_auc"]
)
comparison = pd.DataFrame(
    {
        "Metric": list(untuned_test.keys()),
        "Untuned": list(untuned_test.values()),
        "Tuned": list(tuned_test.values()),
    }
)
comparison["Difference"] = comparison["Tuned"] - comparison["Untuned"]

joblib.dump(tuned_model, MODELS / "best_tuned_model.pkl")
with open(RESULTS / "best_model_parameters.json", "w") as handle:
    json.dump(
        {
            "model": "RandomForestClassifier",
            "imbalance_strategy": selected_strategy,
            "best_parameters": search.best_params_,
            "primary_metric": "average_precision",
            "search_cv": 3,
            "final_validation_cv": 5,
            "random_state": 42,
            "test_used_only_for_final_evaluation": True,
        },
        handle,
        indent=2,
    )
pd.DataFrame(search.cv_results_).to_csv(RESULTS / "hyperparameter_search_results.csv", index=False)
comparison.to_csv(RESULTS / "phase8_tuned_vs_untuned_test.csv", index=False)

final_rows = []
for name in ["SGD Baseline", "Random Forest", "Gradient Boosting", "XGBoost Untuned", "XGBoost Tuned"]:
    row = phase6.loc[phase6["Model"] == name]
    if len(row):
        final_rows.append(
            {
                "Model": name,
                "Strategy": "Original",
                **{key: float(row.iloc[0][key]) for key in ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]},
                **{key: int(row.iloc[0][key]) for key in ["TN", "FP", "FN", "TP"]},
            }
        )
final_rows.append(
    {
        "Model": "Random Forest Tuned",
        "Strategy": selected_strategy,
        **{key: tuned_test[key] for key in ["Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]},
        **{key: tuned_test[key] for key in ["TN", "FP", "FN", "TP"]},
    }
)
pd.DataFrame(final_rows).to_csv(RESULTS / "final_model_comparison_phase8.csv", index=False)

summary = {
    "fraud_percent": float(shares.get(1, 0) * 100),
    "legitimate_percent": float(shares.get(0, 0) * 100),
    "imbalance_ratio": ratio,
    "selected_strategy": selected_strategy,
    "best_parameters": search.best_params_,
    "cv_summary_3_fold": cv_summary,
    "cv_std_3_fold": cv_std,
    "cv_summary_5_fold_training_only": cv_final_summary,
    "cv_std_5_fold_training_only": cv_final_std,
    "final_validation_cv": 5,
    "final_validation_executed": True,
    "readiness_decision_based_on_test": False,
    "tuned_test": tuned_test,
    "untuned_test": untuned_test,
    "train_tuned": train_tuned,
    "train_test_f1_gap": float(train_tuned["F1"] - tuned_test["F1"]),
    "improved_by_rule": bool(improved),
    "decision": "READY FOR THRESHOLD OPTIMIZATION" if improved else "NOT READY — INVESTIGATE MODEL FIRST",
}
with open(RESULTS / "phase8_fast_summary.json", "w") as handle:
    json.dump(summary, handle, indent=2)

print("Saved Phase 8 outputs to:", RESULTS, flush=True)
print("Decision:", summary["decision"], flush=True)
print("Phase 8 fast run complete.", flush=True)


Saved Phase 8 outputs to: /home/ubuntu/fraud_local/results


Decision: READY FOR THRESHOLD OPTIMIZATION


Phase 8 fast run complete.


## Phase 8 conclusion

The readiness decision is based on five-fold training-only validation. Review the saved metrics and report before proceeding.